In [8]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression
import json

# Loading Embeddings

In [9]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [11]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [12]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [13]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-03-12 14:08:13,381 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-12 14:08:47,152 - BERTopic - Dimensionality - Completed ✓
2026-03-12 14:08:47,154 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-12 14:08:55,635 - BERTopic - Cluster - Completed ✓
2026-03-12 14:08:55,669 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 14:09:16,445 - BERTopic - Representation - Completed ✓
2026-03-12 14:10:24,012 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-12 14:11:00,093 - BERTopic - Dimensionality - Completed ✓
2026-03-12 14:11:00,096 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-12 14:11:08,721 - BERTopic - Cluster - Completed ✓
2026-03-12 14:11:08,745 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 14:11:31,134 - BERTopic - Representation - Completed ✓
2026-03-12 14:12:11,381 - BERTopic - D

[(20, np.float64(0.4389079753914821), 732), (50, np.float64(0.4727998565408436), 371), (100, np.float64(0.5028973713641616), 203)]


In [14]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.438908,732
1,50,0.472800,371
2,100,0.502897,203


In [15]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-03-12 14:13:45,986 - BERTopic - Topic reduction - Reducing number of topics
2026-03-12 14:13:46,065 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-12 14:14:05,128 - BERTopic - Representation - Completed ✓
2026-03-12 14:14:05,151 - BERTopic - Topic reduction - Reduced number of topics from 204 to 100
2026-03-12 14:14:08,474 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-12 14:14:08,875 - BERTopic - Dimensionality - Completed ✓
2026-03-12 14:14:08,876 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-12 14:14:26,694 - BERTopic - Cluster - Completed ✓


In [16]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [17]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [18]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [19]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
84    0.288289
21    0.067302
40    0.055859
43    0.054140
5     0.042235
83    0.041068
4     0.038685
42    0.034185
49    0.028368
25    0.026522
dtype: float64

📉 Declining topics:
70   -0.256705
82   -0.273354
80   -0.279917
89   -0.281094
94   -0.289635
73   -0.291879
74   -0.311382
51   -0.318631
77   -0.325920
33   -0.331126
dtype: float64


In [20]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
83    0.120149
4     0.073950
90    0.072506
21    0.069594
88    0.036190
78    0.034457
79    0.028620
37    0.024294
57    0.021735
38    0.021680
dtype: float64

📉 Declining topics:
80   -0.058749
32   -0.064159
74   -0.065734
77   -0.087404
98   -0.088786
93   -0.089504
73   -0.090981
70   -0.097738
51   -0.110499
33   -0.137276
dtype: float64


In [21]:
len(log_trend_scores_12), len(log_trend_scores_24)

(91, 96)

#  Visualization

In [22]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [23]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [24]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [25]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [26]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [27]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [28]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [29]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [30]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
0,9652,0_policy_reinforcement_learning_rl,"[policy, reinforcement, learning, rl, robot, d...",[Representation Learning for Continuous Action...,Reinforcement learning rl,-0.002366,-0.003526,📉 Declining,0.001160,-0.021707,34.192356
1,7472,1_speech_audio_asr_recognition,"[speech, audio, asr, recognition, music, speak...","[Talk, Don't Write: A Study of Direct Speech-B...",Audio asr recognition,-0.001506,-0.000182,📉 Declining,-0.001325,-0.013436,34.279789
2,6089,2_models_preference_explanations_language,"[models, preference, explanations, language, t...",[Shallow Preference Signals: Large Language Mo...,Preference explanations language,-0.013661,-0.011857,📉 Declining,-0.001804,-0.119050,33.163388
3,4258,3_medical_clinical_biomedical_patient,"[medical, clinical, biomedical, patient, healt...",[Refine Medical Diagnosis Using Generation Aug...,Clinical biomedical patient,-0.016749,0.002614,📈 Strong but Slowing,-0.019363,-0.139966,32.942297
4,3854,4_reasoning_cot_mathematical_llms,"[reasoning, cot, mathematical, llms, models, l...",[Reinforcement Learning with Verifiable Reward...,Cot mathematical llms,0.038685,0.073950,🚀 Strong & Accelerating,-0.035265,0.319430,37.798381


In [31]:
topics_df["label"].nunique()

96

In [32]:
def get_topic_words(topic_id, top_n=10, model=reduced_model):
    
    words = model.get_topic(topic_id)
    
    words = words[:top_n]
    
    terms = [w[0] for w in words]
    scores = [w[1] for w in words]
    
    return terms, scores

In [33]:
topic_words = {}
for topic_id in topics_df.index:
    terms, scores = get_topic_words(topic_id)
    topic_words[topic_id] = (terms, scores)  

In [34]:
with open('topic_words.json', 'w') as f:
    json.dump(topic_words, f)

In [35]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [36]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [37]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [38]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [39]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [40]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [41]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [42]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [43]:
top10_impact.to_csv("top10_impact_topics.csv")

In [54]:
raw_data = pd.read_csv("arxiv_abstracts_final.csv").drop_duplicates()

In [55]:
raw_data.head()

,paper_id,title,abstract,published,categories
0,http://arxiv.org/abs/2101.00117v1,Multi-task Retrieval for Knowledge-Intensive T...,Retrieving relevant contexts from a large corp...,2021-01-01,['cs.CL']
1,http://arxiv.org/abs/2101.00121v2,WARP: Word-level Adversarial ReProgramming,Transfer learning from pretrained language mod...,2021-01-01,['cs.CL']
2,http://arxiv.org/abs/2101.00123v2,Intent Classification and Slot Filling for Pri...,Understanding privacy policies is crucial for ...,2021-01-01,['cs.CL']
3,http://arxiv.org/abs/2101.00124v3,Discourse-level Relation Extraction via Graph ...,The ability to capture complex linguistic stru...,2021-01-01,"['cs.CL', 'cs.AI', 'cs.LG']"
4,http://arxiv.org/abs/2101.00130v1,Sensei: Self-Supervised Sensor Name Segmentation,"A sensor name, typically an alphanumeric strin...",2021-01-01,['cs.CL']


In [65]:
rep_docs = reduced_model.get_representative_docs()

rows = []

for topic_id, docs in rep_docs.items():
    for rant, text in enumerate(docs, start=1):
        paper_id = df[df['text'] == text]['paper_id']
        paper_data = raw_data[raw_data['paper_id'] == paper_id.values[0]]
        rows.append({
            "topic_id": topic_id,
            "rank": rant,
            "text": text[:200],
            "title": paper_data['title'].values[0],
            "abstract": paper_data['abstract'].values[0],
            "paper_id": paper_data['paper_id'].values[0],
            "published": paper_data['published'].values[0]
        })
rep_docs_df = pd.DataFrame(rows)
rep_docs_df.to_csv("representative_docs.csv", index=False)